In [17]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Windows


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = r'F:\bmi\cohort8\DON-014372\day0\calibration\Image_001_001.raw'


# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)

# 
print ("DONE...")

memmap :  (90000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (446, 90000)
         self.Fneu (neuropile):  (446, 90000)
         self.iscell (Suite2p cell classifier output):  (602, 2)
              of which number of good cells:  (446,)
         self.spks (deconnvoved spikes):  (446, 90000)
         self.stat (footprints structure):  (446,)
         mean std over all cells :  40.5
# of footprints;  446
DONE...


In [41]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'

#
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells

# 


computing roi traces for SNR indexing: 100%|████████████████████████████████████████| 9000/9000 [02:09<00:00, 69.71it/s]


In [48]:
#########################################################
########### VISUALIZE CELLS BY SNR OR F0 ##################
#########################################################
#
bmi_c.scale=.001
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

bmi_c.min_f0 = 300
# visualize traces
bmi_c.vmin=0
bmi_c.vmax=500
bmi_c.max_n_cells = 50
bmi_c.visualize_traces_snr_order(bmi_c.std_map)


memmap :  (90000, 512, 512)


In [54]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [4,42]
bmi_c.ensemble2 = [21,15]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
# bmi_c.show_traces_ids(bmi_c.both)
# # ######################################################################
# # ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# # ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles2(bmi_c.std_map)

print ("DONE...")


all cells: [ 4 42 21 15]


100%|███████████████████████████████████████████████████████████████████████████| 90000/90000 [00:31<00:00, 2867.59it/s]


Contour:  [226 365]
cell ids:  [ 4 42 21 15]
DONE...


In [51]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# if using binning
bmi_c.binning_flag = True
bmi_c.binning_time = 0.200
bmi_c.smoothing_n_bins = 3

# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 10  <------------------------------------------------------------------
bmi_c.post_reward_lockout_baseline_min = .3 # want to wait until we drop to 30% of threshold <-------------------------------
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
stepper = 0.85
bmi_c.find_reward_thresholds_high_realtime(stepper)  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


####################################
# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


nsec recording:  3000 max # of random rewards (i.e. every 30sec)  100
 @0.25% reward rate:  25
 high guess:  4.696443611308414
updated rewards #:  0  for threshold:  4.696443611308414
updated rewards #:  1  for threshold:  3.991977069612152
updated rewards #:  1  for threshold:  3.393180509170329
updated rewards #:  2  for threshold:  2.88420343279478
updated rewards #:  6  for threshold:  2.4515729178755628
updated rewards #:  9  for threshold:  2.0838369801942282
updated rewards #:  11  for threshold:  1.771261433165094
updated rewards #:  12  for threshold:  1.50557221819033
updated rewards #:  13  for threshold:  1.2797363854617805
updated rewards #:  13  for threshold:  1.0877759276425134
updated rewards #:  13  for threshold:  0.9246095384961364
updated rewards #:  13  for threshold:  0.7859181077217159
updated rewards #:  14  for threshold:  0.6680303915634584
updated rewards #:  14  for threshold:  0.5678258328289396
updated rewards #:  15  for threshold:  0.48265195790459864
u

In [52]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
 couldn't save bmi_c.object .... TO FIX!
Done...
